## Add Demodata

In [1]:
import sys
sys.path.append('../')

In [2]:
from dotenv import load_dotenv
from langchain_community.vectorstores.elasticsearch import ElasticsearchStore

from datasets import load_dataset, Dataset
from datasets import get_dataset_config_names
from mylibs.classes.AppSettings import AppSettings
settings = AppSettings()

c:\Projekte\OTOBO\OTOBO-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### german Q/A data for testing from HuggingFace

In [3]:
name_ds = "deepset/germanquad"
configs = get_dataset_config_names(name_ds)
print(configs)

demodata: Dataset = load_dataset(name_ds, split='test')  # type: ignore

print(demodata.num_rows)
print(demodata.shape) # Shape of the dataset (number of columns, number of rows).
print(demodata.column_names)
print(demodata.features)

['plain_text']
2204
(2204, 4)
['id', 'context', 'question', 'answers']
{'id': Value(dtype='int32', id=None), 'context': Value(dtype='string', id=None), 'question': Value(dtype='string', id=None), 'answers': Sequence(feature={'text': Value(dtype='string', id=None), 'answer_start': Value(dtype='int32', id=None)}, length=-1, id=None)}


### slice into pieces of 50 

In [4]:
import json

n = 0

steps = 100

ab = n * steps
bis = (n+1) * steps

slice = demodata[ab:bis]
# !!! len(slice) is 4 because ['id', 'context', 'question', 'answers']

# slice = dataset[anz:]

datas = []
for i in range(bis-ab):
  data = {}
  data["id"] = slice["id"][i]
  data["process_id"]= f"Slice_{ab}_{bis}"
  data["gdpr_id"]= str(slice["id"][i])
  data["topic"]= slice["question"][i]
  data["type"]= "answer"

  document = f'Q: {slice["question"][i]}\nA: {slice["answers"][i]["text"][0]}\nTopic: {slice["context"][i]}'
  data["document"]= document
  data["len"] = len(document)

  datas.append(data)

In [9]:
len(datas)
# data

100

In [10]:
datas[0]["process_id"]

'Slice_0_100'

In [5]:
import os
import requests

api_url = "http://127.0.0.1:8000/ai/embedding/insert/"

headers = {
  'Content-Type': 'application/json',
  'access_token': settings.AI_API_KEY
}

response = requests.put(api_url, json=datas, headers=headers)
if response.status_code != 200:
  print("Error at: ", response.json)

In [ ]:
from langchain_community.vectorstores.elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings

from mylibs.embedding.embedding import get_vectorstore

embedding = OpenAIEmbeddings()
elastic_vector_search = ElasticsearchStore(
    index_name="test_index",
    embedding=embedding,
    es_cloud_id="the_cloud_id",
    es_user="elastic",
    es_password="your_Password",
    es_api_key="your_apikey",
)

ValidationError: 1 validation error for OpenAIEmbeddings
__root__
  Did not find openai_api_key, please add an environment variable `OPENAI_API_KEY` which contains it, or pass `openai_api_key` as a named parameter. (type=value_error)

In [ ]:
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./demodata.txt", encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings()

In [ ]:
docs[0]


In [ ]:
elastic_vector_search.add_documents(docs)

In [ ]:
import os
import requests

api_url = "http://127.0.0.1:8000/ai/embedding/insert/"

headers = {
  'Content-Type': 'application/json',
  'access_token': settings.AI_API_KEY
}

response = requests.put(api_url, json=datas, headers=headers)
if response.status_code != 200:
  print("Error at: ", response.json)

In [ ]:
print(response.json())
len(response.json())

In [ ]:
import pandas as pd

df = pd.DataFrame(data)
print(df)

In [ ]:
from elasticsearch import Elasticsearch

client = Elasticsearch(
  "https://...",
  api_key=""
)

In [ ]:
# API key should have cluster monitor rights
client.info()